# Multi-PDF -> Markdown (cola por Google Sheet, una sola celda)

Convierte un lote de PDFs a Markdown con [marker-pdf](https://github.com/datalab-to/marker), coordinando varias cuentas de Colab via un Google Sheet (`PDF2MD/Status_PDF`) para que trabajen en paralelo sin pisarse.

**Carpetas:** entrada `MyDrive/PDF2MD/`, salida `MyDrive/PDF2MD/Output/` (un `.zip` por PDF: `.md` + imagenes + `_meta.json`).

**Antes de correr:** `Runtime -> Change runtime type -> T4 GPU`.

## Que hace (todo en una celda)

1. Autoriza **Drive + Sheets al inicio** (los 2 popups primero), antes de instalar marker (~minutos) — autorizas y se queda corriendo.
2. Abre el Sheet **por nombre** (`Status_PDF`) y toma la primera fila reclamable cuyo PDF tenga subido: `TODO`, `DOING` huerfano, o `FAILED` con < 3 intentos.
3. Convierte. **PDF > 100 paginas:** lo trocea automaticamente (la ultima tanda se ajusta sola). **Pagina densa que revienta la VRAM (CUDA OOM):** reintenta ese PDF con batch sizes reducidos.
4. Guarda el `.zip` en `Output/`, marca `DONE`. Reanudable (salta los que ya tienen `.zip`).
5. Mientras procesa un PDF largo, un *heartbeat* refresca su claim cada 5 min (asi no se lo roba otra cuenta por mas que tarde). Si una cuenta se cae, su claim vence a los 15 min y otra lo retoma.
6. Sin trabajo: pita y espera; a los 10 chequeos vacios desconecta el runtime para no gastar cuota.

> **Sheet:** una pestaña con encabezados `pdf | status | worker | claim_time | tries` en la fila 1 y los nombres de los PDFs en `A2:...`. Compartido con las cuentas (basta dejarlo en `PDF2MD/`, ya compartida).


In [ ]:
# ===== Multi-PDF -> MD | cola por Sheet | defensas de VRAM | una sola celda =====
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # antes de torch

import re, shutil, json, time, random, threading, uuid
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
from IPython.display import Audio, display

# ---- Autorizaciones PRIMERO (los 2 popups al inicio) ----
from google.colab import drive
drive.mount("/content/drive")
from google.colab import auth
auth.authenticate_user()
get_ipython().system("pip install -q --upgrade gspread pypdf")
import gspread
from gspread import Cell
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

# Abrir el Sheet POR NOMBRE (si tuvieras dos con el mismo nombre, pega el ID abajo)
SHEET_NAME = "Status_PDF"
SHEET_ID   = ""   # opcional: ID de la URL; si lo pones, se usa en vez del nombre
_sh = gc.open_by_key(SHEET_ID) if SHEET_ID else gc.open(SHEET_NAME)
try:
    ws = _sh.worksheet("Hoja 1")
except Exception:
    ws = _sh.get_worksheet(0)   # primera pestaña como fallback
print(f"Sheet conectado: {_sh.title}")

# ---- Instalar marker (largo la primera vez) ----
get_ipython().system("pip install -q marker-pdf")
import torch
from pypdf import PdfReader
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser
from marker.output import text_from_rendered
from marker.settings import settings

# ---- Carpetas ----
SOURCE_DIR = Path("/content/drive/MyDrive/PDF2MD")
DEST_DIR   = SOURCE_DIR / "Output"
DEST_DIR.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/work")

# ---- Config ----
FORCE_OCR       = True
PAGES_PER_CHUNK = 100
BATCH_NORMAL = {"recognition_batch_size": 96, "detection_batch_size": 24,
                "layout_batch_size": 24, "equation_batch_size": 24, "table_rec_batch_size": 24}
BATCH_LOW    = {"recognition_batch_size": 16, "detection_batch_size": 4,
                "layout_batch_size": 4, "equation_batch_size": 4, "table_rec_batch_size": 4}
TTL_MIN     = 15        # claim huerfano -> reclamable (refrescado por heartbeat)
HEARTBEAT_S = 5 * 60    # refresco del claim mientras proceso un PDF
VERIFY_S    = 3
IDLE_S      = 60
MAX_IDLE    = 10
MAX_TRIES   = 3
COL_STATUS, COL_WORKER, COL_TIME, COL_TRIES = 2, 3, 4, 5

# ---- Cargar modelos UNA sola vez ----
if "MODELS" not in globals():
    print("Cargando modelos de marker (una sola vez)...")
    settings.OUTPUT_IMAGE_FORMAT = "png"
    MODELS = create_model_dict()
    print("Modelos listos.\n")

WORKER = "colab-" + uuid.uuid4().hex[:6]
print(f"Worker: {WORKER}")

# ===== marker: converter + chunking =====
def make_converter(batch_sizes, page_range=None):
    cfg = {"output_format": "markdown", "force_ocr": FORCE_OCR, **batch_sizes}
    if page_range is not None:
        cfg["page_range"] = page_range
    cp = ConfigParser(cfg)
    return PdfConverter(config=cp.generate_config_dict(), artifact_dict=MODELS,
                        processor_list=cp.get_processors(), renderer=cp.get_renderer(),
                        llm_service=cp.get_llm_service())

def plan_chunks(n_pages, chunk_size):
    return [(s, min(s + chunk_size - 1, n_pages - 1)) for s in range(0, n_pages, chunk_size)]

_IMG_REF = re.compile(r"!\[([^\]]*)\]\(([^)]+)\)")
def _prefix_images(text, images, prefix):
    rename = {name: f"{prefix}{name}" for name in (images or {})}
    if not rename:
        return text, {}
    text2 = _IMG_REF.sub(lambda m: f"![{m.group(1)}]({rename.get(m.group(2), m.group(2))})", text)
    return text2, {rename[k]: v for k, v in images.items()}

def convert_pdf(pdf_path, batch_sizes):
    """Devuelve (text, images, metadata). Trocea si supera PAGES_PER_CHUNK."""
    n_pages = len(PdfReader(pdf_path).pages)
    if n_pages <= PAGES_PER_CHUNK:
        print(f"   paginas: {n_pages} -> una pasada")
        rendered = make_converter(batch_sizes)(str(pdf_path))
        text, _, images = text_from_rendered(rendered)
        return text, dict(images or {}), getattr(rendered, "metadata", None)

    ranges = plan_chunks(n_pages, PAGES_PER_CHUNK)
    print(f"   paginas: {n_pages} -> {len(ranges)} tanda(s): " + ", ".join(f"{a}-{b}" for a, b in ranges))
    parts_text, parts_images, last_meta = [], {}, None
    for idx, (a, b) in enumerate(ranges, 1):
        t0 = time.time()
        print(f"    tanda {idx}/{len(ranges)} ({a}-{b})...", end=" ", flush=True)
        rendered = make_converter(batch_sizes, page_range=f"{a}-{b}")(str(pdf_path))
        text, _, images = text_from_rendered(rendered)
        if not text or not text.strip():
            raise RuntimeError(f"tanda {idx} ({a}-{b}) no produjo texto")
        tp, ip = _prefix_images(text, images, prefix=f"p{idx}_")
        parts_text.append(tp); parts_images.update(ip)
        last_meta = getattr(rendered, "metadata", None) or last_meta
        torch.cuda.empty_cache()
        print(f"OK ({time.time()-t0:.1f}s, {len(ip)} img)")
    return "\n\n".join(parts_text), parts_images, last_meta

def _save_zip(stem, text, images, metadata):
    item_dir = WORK / stem
    shutil.rmtree(item_dir, ignore_errors=True); item_dir.mkdir(parents=True)
    try:
        (item_dir / f"{stem}.md").write_text(text, encoding="utf-8")
        for name, img in images.items():
            img.save(str(item_dir / name))
        if metadata is not None:
            (item_dir / f"{stem}_meta.json").write_text(
                json.dumps(metadata, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
        tmp_zip = shutil.make_archive(str(WORK / stem), "zip", str(item_dir))
        final_zip = DEST_DIR / f"{stem}.zip"
        part = str(final_zip) + ".part"
        shutil.copy(tmp_zip, part); os.replace(part, str(final_zip)); os.remove(tmp_zip)
    finally:
        shutil.rmtree(item_dir, ignore_errors=True)

def process_and_save(stem, pdf_path):
    """Convierte+guarda. Ante CUDA OOM reintenta con batch reducido. -> 'done'|'failed'."""
    for attempt, batch in enumerate([BATCH_NORMAL, BATCH_LOW], 1):
        try:
            text, images, meta = convert_pdf(pdf_path, batch)
            if not text or not text.strip():
                raise RuntimeError("marker no produjo texto")
            _save_zip(stem, text, images, meta)
            torch.cuda.empty_cache()
            return "done"
        except Exception as e:
            torch.cuda.empty_cache()
            if "out of memory" in str(e).lower() and attempt == 1:
                print(f"   CUDA OOM con batch normal -> reintento con batch reducido")
                continue
            print(f"   x ERROR: {e}")
            return "failed"
    return "failed"

# ===== Sheet: cola coordinada =====
PDF_EXTS = (".pdf",)
def _now(): return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
def _stem(name):
    name = name.strip()
    return name[:-4] if name.lower().endswith(".pdf") else name
def _orphan(ctime):
    try:
        age = (datetime.now(timezone.utc) -
               datetime.strptime(ctime, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)).total_seconds() / 60
        return age > TTL_MIN
    except Exception:
        return True
def _tries(row): return int(row[4]) if len(row) > 4 and row[4].strip().isdigit() else 0
def beep(freq=880, dur=0.12):
    sr = 8000; t = np.linspace(0, dur, int(sr*dur), endpoint=False)
    display(Audio((0.2*np.sin(2*np.pi*freq*t)).astype(np.float32), rate=sr, autoplay=True))
def _heartbeat(rownum, stop):
    while not stop.wait(HEARTBEAT_S):
        try: ws.update_cells([Cell(rownum, COL_TIME, _now())])
        except Exception: pass

local_stems = {p.stem for p in sorted(SOURCE_DIR.glob("*.pdf"))}
print(f"{len(local_stems)} PDF(s) en PDF2MD\n")

done = skipped = failed = 0
idle = 0
t_global = time.time()

while True:
    rows = ws.get_all_values()[1:]
    target = None
    for i, row in enumerate(rows, start=2):
        name   = row[0] if len(row) > 0 else ""
        status = (row[1] if len(row) > 1 else "").strip().upper()
        ctime  = row[3] if len(row) > 3 else ""
        if not name:                                 continue
        if _stem(name) not in local_stems:           continue
        if status == "DONE":                         continue
        if status == "DOING" and not _orphan(ctime): continue
        if status == "FAILED" and _tries(row) >= MAX_TRIES: continue
        target = (i, _stem(name), name, _tries(row))
        break

    if target is None:
        pend = 0
        for r in rows:
            if not (r and r[0]): continue
            st = (r[1] if len(r) > 1 else "").strip().upper()
            if st == "DONE": continue
            if st == "FAILED" and _tries(r) >= MAX_TRIES: continue
            pend += 1
        if pend == 0:
            print("\nTodo terminado."); break
        idle += 1; beep()
        if idle >= MAX_IDLE:
            print(f"\n{MAX_IDLE} chequeos sin trabajo -> desconecto runtime.")
            from google.colab import runtime; runtime.unassign(); break
        print(f"\nNada para mi ahora ({pend} pendientes). Idle {idle}/{MAX_IDLE}, espero {IDLE_S}s...")
        time.sleep(IDLE_S); continue

    idle = 0
    rownum, stem, name, tries = target

    if (DEST_DIR / f"{stem}.zip").exists():
        ws.update_cells([Cell(rownum, COL_STATUS, "DONE")]); skipped += 1; continue

    ws.update_cells([Cell(rownum, COL_STATUS, "DOING"), Cell(rownum, COL_WORKER, WORKER),
                     Cell(rownum, COL_TIME, _now())])
    time.sleep(VERIFY_S)
    check = ws.row_values(rownum)
    if len(check) < 3 or check[2] != WORKER:
        continue

    pdf_path = next((p for p in SOURCE_DIR.glob("*.pdf") if p.stem == stem), None)
    if pdf_path is None:
        ws.update_cells([Cell(rownum, COL_STATUS, "TODO"), Cell(rownum, COL_WORKER, ""),
                         Cell(rownum, COL_TIME, "")]); continue

    print(f"\n[fila {rownum}] (intento {tries+1}/{MAX_TRIES}) {name}")
    t0 = time.time()
    stop = threading.Event()
    hb = threading.Thread(target=_heartbeat, args=(rownum, stop), daemon=True); hb.start()
    try:
        outcome = process_and_save(stem, pdf_path)
    except Exception as e:
        print(f"   x excepcion: {e}"); outcome = "failed"
    finally:
        stop.set()

    if outcome == "done":
        ws.update_cells([Cell(rownum, COL_STATUS, "DONE"), Cell(rownum, COL_WORKER, WORKER),
                         Cell(rownum, COL_TIME, _now())])
        print(f"   ok -> {stem}.zip ({time.time()-t0:.1f}s)")
        done += 1
    else:
        ws.update_cells([Cell(rownum, COL_STATUS, "FAILED"), Cell(rownum, COL_WORKER, WORKER),
                         Cell(rownum, COL_TIME, _now()), Cell(rownum, COL_TRIES, str(tries + 1))])
        failed += 1

print(f"\n=== Fin ({WORKER}) ===  ok: {done}  saltados: {skipped}  fallidos: {failed}")
print(f"  tiempo: {(time.time()-t_global)/60:.1f} min")
sr = 22050; out_audio = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    out_audio = np.concatenate([out_audio, (0.3*np.exp(-3*t)*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out_audio, rate=sr, autoplay=True))
